# iSCORS — Classical Deliverable (γ + apparent α + density), no ML

The honest ACF-line output, computed by a **GPU classical fit in seconds — no network, no
training, no checkpoint** (the U-Net was retired; see `RETROSPECTIVE.md`).

- **γ map** — diffusion-rate map.
- **apparent global α** — one cell-wide anomalous exponent (~0.6); a *temporal-decorrelation*
  exponent (STICS could not verify it as true sub-diffusion — sub-PSF).
- **density G(0)=CV²** — the amplitude channel iSCORS normalises away: high-SNR, the
  cleanest single map (≈ condensation map), 'how much/many' complementary to γ's 'how fast'.

Paths/preprocessing mirror `iscors_real_runner.ipynb`. A Gradio front-end is at the end.


In [ ]:
# ── setup ────────────────────────────────────────────────────────────────
import os, subprocess, sys
REPO='https://github.com/breezy90126/iscors-net.git'; BRANCH='claude/brave-ramanujan-33eps3'
REPO_DIR='/content/iscors-net'
try:
    from google.colab import drive; drive.mount('/content/drive', force_remount=False)
except Exception: pass
if os.path.isdir(REPO_DIR):
    for c in (['git','-C',REPO_DIR,'fetch','origin'],['git','-C',REPO_DIR,'checkout',BRANCH],
              ['git','-C',REPO_DIR,'pull','origin',BRANCH]): subprocess.run(c, check=False)
else:
    subprocess.run(['git','clone','--branch',BRANCH,REPO,REPO_DIR], check=False)
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
subprocess.run(['pip','install','-q','tifffile','scipy','gradio'], check=False)
print('setup done:', os.getcwd())


In [ ]:
# ── config (same paths as iscors_real_runner.ipynb) ──────────────────────
import numpy as np
ZIP_PATH    = '/content/drive/MyDrive/iscors_test/large_file.zip'   # .zip or .tif
EXTRACT_DIR = '/content/real_data'                                  # zip extraction cache
VIDEO_FNAME = 'COBRI_rarw_video.tif'                                # name inside the zip
MAT_FNAME   = 'Output_iSCORS_map.mat'                               # iSCORS GT (for Cond_map check)
N_FRAMES    = 2000
BIN_FACTOR  = 2
RECON_TAUS  = (1, 2, 4, 8, 16, 32, 48, 64, 96, 128)
GAMMA_SCALE = 2.0
CONDENSATION_SLOPE = 3.0      # log-log baseline slope (iSCORS condensation projection)
DSTAR_FROM = 'gamma'          # 1/D*: 'gamma' → 1/γ ; 'tauD' → τ_D = γ^(-1/α)
print('config ready')


In [ ]:
# ── reusable pipeline: load → preprocess → classical analyze ─────────────
import os, sys, numpy as np, tifffile, zipfile, tempfile
from scipy.ndimage import gaussian_filter
# robust path: don't rely on the setup cell's sys.path persisting
REPO_DIR = '/content/iscors-net'
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)
    if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)
import importlib, utils.gpu_iscors_fit as _gf; importlib.reload(_gf)
from utils.gpu_iscors_fit import gpu_fit_maps, compute_density, condensation, condensation_projection

def load_video(path, fname=None, n_frames=2000, bin_factor=2):
    if str(path).lower().endswith('.zip'):
        ed = globals().get('EXTRACT_DIR', '/content/real_data'); os.makedirs(ed, exist_ok=True)
        def _find(root, name):
            for dp, _, fs in os.walk(root):
                if name in fs: return os.path.join(dp, name)
            return None
        vp = _find(ed, fname)            # reuse cached extraction (no re-extract per call)
        if not vp:
            with zipfile.ZipFile(path) as z: z.extractall(ed)
            vp = _find(ed, fname)
        assert vp, f'{fname} not found in zip'
    else:
        vp = path
    H0, W0 = tifffile.imread(vp, key=0).shape
    Hb, Wb = (H0//bin_factor)*1, (W0//bin_factor)*1
    with tifffile.TiffFile(vp) as tf:
        try:    total = int(tf.series[0].shape[0])   # robust for large/BigTIFF
        except Exception: total = len(tf.pages)
    n = min(n_frames, total)
    # crop to a bin-divisible size (avoids reshape misalignment)
    Hc, Wc = Hb*bin_factor, Wb*bin_factor
    raw = np.empty((n, Hb, Wb), np.float32)
    for s in range(0, n, 100):
        e = min(s+100, n); ch = tifffile.imread(vp, key=range(s, e)).astype(np.float32)
        raw[s:e] = ch[:, :Hc, :Wc].reshape(e-s, Hb, bin_factor, Wb, bin_factor).mean((2, 4))
    # ── sanity: a >4GB classic TIFF (offset overflow) or truncated extraction
    #    yields garbage/constant frames → CV≈0 → 'no cell pixels'. Surface it here.
    finite = np.isfinite(raw).all()
    mean_I = raw.mean(0); cvm = raw.std(0) / (np.abs(mean_I) + 1e-10)
    ncell = int((cvm >= 0.005).sum())
    print(f'[load] {os.path.basename(vp)}: {n}/{total} frames  raw {raw.shape}  '
          f'mean={raw.mean():.3g} std={raw.std():.3g} min={raw.min():.3g} max={raw.max():.3g}')
    print(f'[load] cell pixels @CV>=0.005: {ncell}/{Hb*Wb}  ({100*ncell/(Hb*Wb):.1f}%)')
    if (not finite) or raw.std() < 1e-9 or ncell == 0:
        print('[load] *** DEGENERATE DATA *** — almost certainly a TIFF read problem:')
        print('       - >4GB classic TIFF: page offsets overflow at ~4GB (your warning was'
              ' offset≈4.10GB) → frames near/after 4GB read as garbage. Lower N_FRAMES so the'
              ' read stays under 4GB, or re-save the video as BigTIFF.')
        print('       - truncated extraction (disk full): re-extract, check `df -h /content`.')
        print('       Compare with iscors_real_runner.ipynb (same loader) to confirm.')
    return raw

def preprocess(raw):
    ff = raw / (np.median(raw, axis=0)[None] + 1e-10)           # flat-field
    proc = np.empty_like(ff)
    for t in range(len(ff)):
        proc[t] = ff[t] / (gaussian_filter(ff[t], sigma=4) + 1e-10)  # per-frame BG removal
    return proc

def analyze(video_proc):
    out = gpu_fit_maps(video_proc, recon_taus=RECON_TAUS, n_components=1, global_alpha=True,
                       gamma_scale=GAMMA_SCALE, min_cv=0.005, n_steps=500, verbose=False)
    cell = out['cell_mask']
    dens, _ = compute_density(video_proc, min_cv=0.005)             # V_DLS = CV²
    g = np.clip(out['gamma'], 1e-6, None)
    if DSTAR_FROM == 'tauD':
        inv_Dstar = np.power(g, -1.0/np.clip(out['alpha'], 0.1, 2.0))  # τ_D = γ^(-1/α)
    else:
        inv_Dstar = 1.0 / g                                          # 1/D* ∝ 1/γ
    # iSCORS condensation: slope-3 log-log perpendicular projection (V_DLS vs 1/D*)
    cond, cinfo = condensation_projection(dens, inv_Dstar, cell, slope=CONDENSATION_SLOPE)
    return dict(gamma=out['gamma'], alpha_global=float(np.nanmedian(out['alpha'][cell])),
                density=dens, inv_Dstar=inv_Dstar, condensation=cond, cell=cell,
                density_b=cinfo['b'])

def make_fig(data, cmap, title):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(5, 4.5))
    p1, p99 = np.nanpercentile(data, 1), np.nanpercentile(data, 99)
    im = ax.imshow(data, cmap=cmap, vmin=p1, vmax=p99); plt.colorbar(im, ax=ax)
    ax.set_title(title, fontsize=11); ax.axis('off'); fig.tight_layout(); return fig
print('pipeline defined')


In [ ]:
# ── run on the configured file + plot ────────────────────────────────────
import matplotlib.pyplot as plt, time
t0 = time.time()
raw = load_video(ZIP_PATH, VIDEO_FNAME, N_FRAMES, BIN_FACTOR)
video_proc = preprocess(raw)
res = analyze(video_proc)
print(f'done in {time.time()-t0:.1f}s   apparent global α = {res["alpha_global"]:.3f}   '
      f'cell={100*res["cell"].mean():.1f}%   density intercept b = {res["density_b"]:.3f}')
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
for a_, d, cm, ttl in [(ax[0], res['gamma'], 'magma', 'γ (diffusion rate)'),
                       (ax[1], res['density'], 'viridis', 'density  G(0)=CV²'),
                       (ax[2], res['condensation'], 'inferno', 'Condensation (slope-3 proj)')]:
    p1, p99 = np.nanpercentile(d, 1), np.nanpercentile(d, 99)
    im = a_.imshow(d, cmap=cm, vmin=p1, vmax=p99); plt.colorbar(im, ax=a_)
    a_.set_title(ttl, fontsize=11); a_.axis('off')
plt.suptitle(f'iSCORS classical deliverable — apparent global α ≈ {res["alpha_global"]:.2f}', fontsize=13)
plt.tight_layout(); plt.show()


In [ ]:
# ── validate: our Condensation vs the .mat Cond_map (Pearson) ────────────
# Also pits raw CV², raw γ, and both Φ forms against Cond_map → tells which quantity
# the iSCORS condensation map actually is, and whether the CV²/Φ correction helps.
import os, numpy as np, matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from scipy.ndimage import zoom
from utils.gpu_iscors_fit import condensation

def _find(root, name):
    for dp, _, fs in os.walk(root):
        if name in fs: return os.path.join(dp, name)
    return None
mat_path = _find(EXTRACT_DIR, MAT_FNAME)
if not mat_path and str(ZIP_PATH).lower().endswith('.zip'):
    import zipfile
    with zipfile.ZipFile(ZIP_PATH) as z: z.extractall(EXTRACT_DIR)
    mat_path = _find(EXTRACT_DIR, MAT_FNAME)
assert mat_path, f'{MAT_FNAME} not found under {EXTRACT_DIR}'
try:
    import scipy.io; mat = scipy.io.loadmat(mat_path); mat_keys=[k for k in mat if not k.startswith('_')]
except NotImplementedError:
    import h5py; mat={}
    with h5py.File(mat_path,'r') as hf:
        for k in hf: mat[k]=np.array(hf[k])
    mat_keys=list(mat)
assert 'Cond_map' in mat, f'Cond_map not in .mat; keys={mat_keys}'
cond_raw = np.asarray(mat['Cond_map']).astype(np.float64).squeeze()

cell = res['cell']; Hm, Wm = res['condensation'].shape
Hc, Wc = cond_raw.shape
if abs(Hc-Wm) < abs(Hc-Hm) and Hc != Hm: cond_raw = cond_raw.T   # transpose if column-major
def _align(a):
    return (a if (a.shape[0],a.shape[1])==(Hm,Wm)
            else zoom(a,(Hm/a.shape[0],Wm/a.shape[1]),order=1)).astype(np.float32)
def _coord(a):                                   # MATLAB→Python: flipud + rot90 clockwise
    o = np.rot90(np.flipud(a), k=-1).astype(np.float32)
    return o if o.shape==(Hm,Wm) else zoom(o,(Hm/o.shape[0],Wm/o.shape[1]),order=1).astype(np.float32)
cond_gt = _coord(_align(cond_raw)); cond_gt[~cell] = np.nan

# candidate maps to compare against Cond_map
a_full = np.full_like(res['gamma'], res['alpha_global'])
cands = {
    'Condensation (slope-3 proj)': res['condensation'],
    'CV²/γ (simple V_DLS/D)'     : condensation(res['density'], res['gamma'], a_full, blur='gamma'),
    'raw CV² (V_DLS)'            : res['density'],
    '1/γ (1/D*)'                : 1.0/np.clip(res['gamma'], 1e-6, None),
}
print('=== vs .mat Cond_map (cell pixels) ===')
best, best_p = None, -1
for name, m in cands.items():
    ok = cell & np.isfinite(cond_gt) & np.isfinite(m)
    x, y = cond_gt[ok].astype(float), m[ok].astype(float)
    pr = pearsonr(x, y)[0]; sr = spearmanr(x, y).statistic
    print(f'  {name:24s}  Pearson={pr:+.3f}  Spearman={sr:+.3f}  N={ok.sum()}')
    if abs(pr) > best_p: best, best_p = name, abs(pr)
print(f'  → best match to Cond_map: {best} (|Pearson|={best_p:.3f})')

# maps + scatter of the (default) condensation
our = cands['Condensation (slope-3 proj)']
ok = cell & np.isfinite(cond_gt) & np.isfinite(our)
pr = pearsonr(cond_gt[ok], our[ok])[0]; sr = spearmanr(cond_gt[ok], our[ok]).statistic
fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
for a_, d, t in [(ax[0], cond_gt, '.mat Cond_map'),
                 (ax[1], our, 'our Condensation (slope-3 proj)')]:
    p1, p99 = np.nanpercentile(d, 1), np.nanpercentile(d, 99)
    im = a_.imshow(d, cmap='inferno', vmin=p1, vmax=p99); plt.colorbar(im, ax=a_, fraction=0.046)
    a_.set_title(t, fontsize=11); a_.axis('off')
ax[2].scatter(cond_gt[ok], our[ok], alpha=0.05, s=1, c='steelblue')
ax[2].set_xlabel('.mat Cond_map'); ax[2].set_ylabel('our Condensation')
ax[2].set_title(f'Pearson={pr:+.3f}  Spearman={sr:+.3f}')
fig.suptitle('Condensation validation vs iSCORS .mat Cond_map', fontsize=13)
plt.tight_layout(); plt.show()
if 'SAVE_DIR' in globals():
    fig.savefig(os.path.join(SAVE_DIR, 'condensation_vs_condmap.png'), dpi=120, bbox_inches='tight')


In [ ]:
# ── Stage A: classical noise-free V_DLS (amplitude A) vs raw CV²=G(0) ─────
# Raw V_DLS=G(0) includes the τ=0 noise spike; A extrapolates the noise-free τ≥1 ACF
# to τ→0. Does removing that noise bias improve the condensation (vs Cond_map)? If A≈raw
# the bias is small (no room for FAST); if A clearly beats raw, ML frame-denoising (Stage B)
# is worth trying. Needs cond_gt from the Cond_map validation cell above.
import numpy as np, matplotlib.pyplot as plt, os
from scipy.stats import pearsonr, spearmanr
from utils.gpu_iscors_fit import vdls_amplitude, condensation_projection

cell = res['cell']
a_full = np.full_like(res['gamma'], res['alpha_global'])
A   = vdls_amplitude(video_proc, RECON_TAUS, res['gamma'], a_full)   # noise-free V_DLS
cv2 = res['density']                                                 # raw V_DLS = G(0)
ratio = cv2 / np.clip(A, 1e-12, None)
print(f'V_DLS  raw/clean ratio (median over cell) = {np.nanmedian(ratio[cell]):.2f}  '
      f'(>1 ⇒ G(0) inflated by τ=0 noise; ≈1 ⇒ little noise to remove)')

inv_D = 1.0 / np.clip(res['gamma'], 1e-6, None)
cond_clean, _ = condensation_projection(A,   inv_D, cell, slope=CONDENSATION_SLOPE)
cond_raw      = res['condensation']                                  # from raw CV²

if 'cond_gt' in globals():
    print('\n=== vs .mat Cond_map (does noise-removal help?) ===')
    for nm, m in [('condensation (raw CV²)', cond_raw),
                  ('condensation (clean A)', cond_clean),
                  ('raw CV² (V_DLS)',        cv2),
                  ('clean A (V_DLS)',        A)]:
        ok = cell & np.isfinite(cond_gt) & np.isfinite(m)
        pr = pearsonr(cond_gt[ok], m[ok])[0]; sr = spearmanr(cond_gt[ok], m[ok]).statistic
        print(f'  {nm:24s}  Pearson={pr:+.3f}  Spearman={sr:+.3f}')
    print('  VERDICT: clean ≫ raw ⇒ noise bias matters → Stage B (FAST) worth it;'
          ' clean ≈ raw ⇒ V_DLS already clean, ML not needed.')
else:
    print('(run the Cond_map validation cell first to define cond_gt for comparison)')

fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
for a_, d, t in [(ax[0], cv2,        'raw V_DLS = CV² = G(0)'),
                 (ax[1], A,          'clean V_DLS = A (ACF τ→0)'),
                 (ax[2], cond_clean, 'Condensation (clean V_DLS)')]:
    p1, p99 = np.nanpercentile(d, 1), np.nanpercentile(d, 99)
    im = a_.imshow(d, cmap='inferno', vmin=p1, vmax=p99); plt.colorbar(im, ax=a_, fraction=0.046)
    a_.set_title(t, fontsize=11); a_.axis('off')
fig.suptitle('Stage A — classical noise-free V_DLS (amplitude extrapolation)', fontsize=13)
plt.tight_layout(); plt.show()
if 'SAVE_DIR' in globals():
    fig.savefig(os.path.join(SAVE_DIR, 'stageA_vdls_clean.png'), dpi=120, bbox_inches='tight')


In [ ]:
# ── Gradio front-end (layout preview) ────────────────────────────────────
import gradio as gr
def _run(file, bin_factor, n_frames):
    if file is None: raise gr.Error('請上傳 .tif（或含 tif 的 .zip）')
    fname = VIDEO_FNAME if str(file).lower().endswith('.zip') else None
    raw = load_video(file, fname, int(n_frames), int(bin_factor))
    res = analyze(preprocess(raw))
    stats = (f'cell pixels      : {100*res["cell"].mean():.1f}%\n'
             f'apparent global α: {res["alpha_global"]:.3f}  (α=1 ⇒ normal)\n'
             f'γ  median(cell)  : {np.nanmedian(res["gamma"][res["cell"]]):.3f}\n'
             f'density median   : {np.nanmedian(res["density"][res["cell"]]):.4f}  (CV²)')
    return (make_fig(res['gamma'],'magma','γ (diffusion rate)'),
            make_fig(res['density'],'viridis','density  G(0)=CV²'),
            make_fig(res['condensation'],'inferno','Condensation (slope-3 proj)'), stats)

with gr.Blocks(title='iSCORS classical (γ + α + density)') as demo:
    gr.Markdown('## iSCORS classical deliverable — GPU fit, no ML\n'
                'γ map + apparent global α + density G(0)=CV². Seconds, no training, no checkpoint.')
    with gr.Row():
        with gr.Column(scale=1):
            f_in = gr.File(label='影片 (.tif 或含 tif 的 .zip)', file_types=['.tif','.tiff','.zip'], type='filepath')
            b_in = gr.Slider(label='BIN_FACTOR', minimum=1, maximum=8, step=1, value=2)
            n_in = gr.Number(label='N_FRAMES', value=2000, precision=0)
            run  = gr.Button('分析', variant='primary')
        with gr.Column(scale=2):
            with gr.Tabs():
                with gr.Tab('γ 擴散速率'):       g_out = gr.Plot()
                with gr.Tab('密度 G(0)=CV²'):    d_out = gr.Plot()
                with gr.Tab('Condensation'):    c_out = gr.Plot()
            s_out = gr.Textbox(label='統計摘要', lines=5)
    run.click(_run, inputs=[f_in, b_in, n_in], outputs=[g_out, d_out, c_out, s_out])

demo.launch(share=True)
